# 01 - Analise Exploratoria de Dados (EDA)

Objetivo: caracterizar o dataset DAiSEE antes de qualquer modelagem: distribuicao de classes (Engagement, Boredom, Confusion, Frustration), duracao/resolucao dos videos, "buracos" mencionados no README original, correlacao entre rotulos e possiveis efeitos por sujeito.

Este notebook alimenta diretamente o Objetivo Especifico 1 (indicadores mais relacionados) e a secao 'Estado da Arte'/'Introducao' do docx final (ver Secao 11 do plano).

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

ROOT = Path.cwd().parent
LABELS_DIR = ROOT / "datasets" / "DAiSEE" / "Labels"
VIDEOS_DIR = ROOT / "datasets" / "DAiSEE" / "DataSet"

LABEL_COLS = ["Boredom", "Engagement", "Confusion", "Frustration"]
LEVEL_NAMES = {0: "Muito Baixo", 1: "Baixo", 2: "Alto", 3: "Muito Alto"}

In [ ]:
def load_labels(name):
    df = pd.read_csv(LABELS_DIR / name)
    df.columns = [c.strip() for c in df.columns]
    df["split"] = name.replace("Labels.csv", "")
    return df

train_df = load_labels("TrainLabels.csv")
val_df = load_labels("ValidationLabels.csv")
test_df = load_labels("TestLabels.csv")
all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

print("Total de clipes:", len(all_df))
all_df.head()

## 1. Distribuicao de classes por rotulo e por split

Verificar quantitativamente o desbalanceamento ja identificado no plano (Engagement: 0,68% / 5,1% / 49,5% / 44,7%).

In [ ]:
for col in LABEL_COLS:
    dist = all_df.groupby("split")[col].value_counts(normalize=True).mul(100).round(2)
    print(f"\n=== {col} (% por split) ===")
    print(dist.unstack(fill_value=0))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, col in zip(axes.ravel(), LABEL_COLS):
    sns.countplot(data=all_df, x=col, hue="split", ax=ax)
    ax.set_title(col)
fig.suptitle("Distribuicao de rotulos por split (evidencia do desbalanceamento)")
fig.tight_layout()
plt.show()

## 2. Correlacao entre rotulos

Ex.: espera-se correlacao negativa entre Engagement e Boredom/Confusion/Frustration.

In [ ]:
corr = all_df[LABEL_COLS].corr(method="spearman")
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlacao (Spearman) entre rotulos ordinais")
plt.show()

## 3. Metadados dos videos (duracao/resolucao/FPS) e deteccao de "buracos"

O README do dataset menciona explicitamente possiveis lacunas na numeracao dos clipes devido a limpeza de dados. Aqui validamos quantos ClipIDs dos CSVs realmente existem em disco.

In [ ]:
def resolve_video_path(clip_id, split):
    user_id = clip_id[:6]
    stub = clip_id.replace(".avi", "")
    return VIDEOS_DIR / split / user_id / stub / clip_id

split_map = {"Train": train_df, "Validation": val_df, "Test": test_df}
missing_report = {}
for split_name, df in split_map.items():
    paths = [resolve_video_path(cid, split_name) for cid in df["ClipID"]]
    missing = sum(not p.exists() for p in paths)
    missing_report[split_name] = (missing, len(df))
    print(f"{split_name}: {missing}/{len(df)} clipes ausentes em disco")

missing_report

In [ ]:
# Metadados (duracao/fps/resolucao) de uma AMOSTRA de videos (custoso para rodar em todos)
import cv2

def video_meta(path):
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        return None
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    w = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
    h = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    cap.release()
    duration = frame_count / fps if fps else np.nan
    return {"fps": fps, "n_frames": frame_count, "width": w, "height": h, "duration_s": duration}

sample = train_df.sample(n=min(50, len(train_df)), random_state=RANDOM_STATE)
meta_rows = []
for cid in sample["ClipID"]:
    p = resolve_video_path(cid, "Train")
    if p.exists():
        m = video_meta(p)
        if m:
            m["ClipID"] = cid
            meta_rows.append(m)

meta_df = pd.DataFrame(meta_rows)
meta_df.describe()

## 4. Analise por sujeito (heterogeneidade entre estudantes)

Verificar se alguns sujeitos concentram desproporcionalmente rotulos extremos (relevante para a discussao de ruido de rotulo, ver referencia Vedernikov 2026 na Secao 3 do plano).

In [ ]:
all_df["subject_id"] = all_df["ClipID"].str[:6]
subject_engagement = all_df.groupby("subject_id")["Engagement"].mean().sort_values()
print("Sujeitos com media de Engagement mais baixa:")
print(subject_engagement.head(10))
print("\nSujeitos com media de Engagement mais alta:")
print(subject_engagement.tail(10))

## Checklist de saida
- [ ] Tabelas/graficos de distribuicao de classes gerados (usar no docx/slides, Secoes 11-12 do plano)
- [ ] Matriz de correlacao entre rotulos gerada
- [ ] Taxa de clipes ausentes em disco documentada (limitacao a citar)
- [ ] Estatisticas de duracao/resolucao/FPS coletadas (amostra)
- [ ] Analise por sujeito registrada como insumo para a discussao de ruido de rotulo

Proximo passo: `02_preprocessamento_landmarks.ipynb`.